# Predicting Sale Amounts with Spark MLlib

**Business question**: Given a product, quantity, salesperson profile, and sales region, can we accurately predict the dollar amount of a sale?

This notebook demonstrates the **complete MLlib pipeline** end-to-end:

1. Load and join multiple datasets
2. Explore the data
3. Engineer features (encoding, assembly)
4. Train a baseline model (Linear Regression)
5. Evaluate performance (RMSE, R²)
6. Build an improved model (Gradient-Boosted Trees)
7. Tune hyperparameters with cross-validation
8. Compare models and visualize results
9. **Scale up**: Generate 500K rows and prove Spark’s parallelism
10. Save and reload the production model

**Data**: 100 sales transactions, 30 employees across 6 departments. Three CSV files joined into a single feature-rich dataset.

In [1]:
%matplotlib inline
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, datediff, lit, rand, floor, element_at, array, expr,
    round as spark_round
)
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("10 - MLlib Demo: Sales Prediction") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


---
## 1. Load and Join Datasets

We have three related datasets: **sales** transactions, **employees** who made them, and **departments** they belong to. Joining them gives us richer features for prediction.

In [2]:
employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

print(f"Employees:   {employees.count()} rows, {len(employees.columns)} columns")
print(f"Departments: {departments.count()} rows, {len(departments.columns)} columns")
print(f"Sales:       {sales.count()} rows, {len(sales.columns)} columns")

Employees:   30 rows, 6 columns
Departments: 6 rows, 4 columns
Sales:       100 rows, 7 columns


In [3]:
# Join sales -> employees -> departments into a single enriched DataFrame
data = (
    sales
    .join(employees.select("employee_id", "salary", "hire_date", "city", "department_id"),
          on="employee_id")
    .join(departments.select("department_id", "department_name", "budget"),
          on="department_id")
    .withColumn("tenure_days", datediff(lit("2024-12-01"), col("hire_date")))
    .drop("hire_date", "sale_date", "sale_id")
)

print(f"Combined dataset: {data.count()} rows x {len(data.columns)} columns\n")
data.printSchema()

Combined dataset: 100 rows x 11 columns

root
 |-- department_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- budget: integer (nullable = true)
 |-- tenure_days: integer (nullable = true)



In [4]:
data.select("product", "quantity", "amount", "region", "city",
            "department_name", "salary", "budget", "tenure_days").show(10, truncate=False)

+--------+--------+------+-------+-------+---------------+------+-------+-----------+
|product |quantity|amount|region |city   |department_name|salary|budget |tenure_days|
+--------+--------+------+-------+-------+---------------+------+-------+-----------+
|Widget A|5       |1250.0|Central|Chicago|Sales          |67000 |1800000|1421       |
|Widget B|3       |890.5 |Central|Chicago|Sales          |71000 |1800000|1467       |
|Gadget X|1       |2100.0|Central|Chicago|Sales          |76000 |1800000|1798       |
|Widget A|2       |625.0 |Central|Chicago|Sales          |67000 |1800000|1421       |
|Gadget Y|2       |1780.0|Central|Chicago|Sales          |73000 |1800000|1336       |
|Widget C|6       |450.0 |Central|Chicago|Sales          |71000 |1800000|1467       |
|Gadget X|1       |2100.0|Central|Chicago|Sales          |74000 |1800000|1625       |
|Widget B|2       |593.67|Central|Chicago|Sales          |76000 |1800000|1798       |
|Widget A|7       |1875.0|Central|Chicago|Sales       

---
## 2. Explore the Data

Before modeling, understand the target variable (`amount`) and feature distributions.

In [5]:
data.select("amount", "quantity", "salary", "budget", "tenure_days").describe().show()

+-------+------------------+------------------+------------------+-----------------+-----------------+
|summary|            amount|          quantity|            salary|           budget|      tenure_days|
+-------+------------------+------------------+------------------+-----------------+-----------------+
|  count|               100|               100|               100|              100|              100|
|   mean|           2068.27|              3.94|           83090.0|        1694000.0|          1872.95|
| stddev|1929.4469783192937|3.1328174946117517|13211.637814342845|525860.5261181835|575.1933601104514|
|    min|             150.0|                 1|             67000|           800000|              937|
|    max|            9600.0|                15|            115000|          2500000|             3458|
+-------+------------------+------------------+------------------+-----------------+-----------------+



In [6]:
from pyspark.sql.functions import avg, count

# Average sale amount by product -- product type clearly influences price
print("Average sale amount by product:")
data.groupBy("product").agg(
    spark_round(avg("amount"), 2).alias("avg_amount"),
    spark_round(avg("quantity"), 1).alias("avg_qty"),
    count("*").alias("num_sales")
).orderBy(col("avg_amount").desc()).show()

# Sales by region
print("Sales count by region:")
data.groupBy("region").agg(count("*").alias("count")).orderBy(col("count").desc()).show()

Average sale amount by product:
+--------+----------+-------+---------+
| product|avg_amount|avg_qty|num_sales|
+--------+----------+-------+---------+
|Gadget Z|   4923.08|    1.5|       13|
|Gadget X|    3300.0|    1.6|       14|
|Gadget Y|   2190.77|    2.5|       13|
|Widget A|   1836.96|    7.1|       23|
|Widget B|    949.85|    3.2|       20|
|Widget C|    405.88|    5.4|       17|
+--------+----------+-------+---------+

Sales count by region:
+-------+-----+
| region|count|
+-------+-----+
|   West|   34|
|   East|   32|
|Central|   22|
|  South|   12|
+-------+-----+



---
## 3. Feature Engineering

Our label (target) is `amount`. Our features:

| Feature | Type | Encoding Strategy |
|---------|------|-------------------|
| `quantity` | Numeric | Use directly |
| `salary` | Numeric | Use directly |
| `budget` | Numeric | Use directly |
| `tenure_days` | Numeric | Use directly |
| `product` | Categorical (6 values) | StringIndexer + OneHotEncoder (linear) / StringIndexer only (tree) |
| `region` | Categorical (4 values) | StringIndexer + OneHotEncoder (linear) / StringIndexer only (tree) |
| `city` | Categorical (3 values) | StringIndexer + OneHotEncoder (linear) / StringIndexer only (tree) |

**Why two encoding strategies?** Linear models interpret numeric indices as ordinal (Widget A=0 < Widget B=1), which is wrong for unordered categories. OneHotEncoding fixes this. Tree-based models split on thresholds, so they handle indices correctly without one-hot encoding.

In [7]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print(f"Training set: {train.count()} rows")
print(f"Test set:     {test.count()} rows")

Training set: 83 rows
Test set:     17 rows


---
## 4. Baseline Model: Linear Regression Pipeline

We start with a simple linear model as our baseline. The pipeline encodes all categoricals with OneHotEncoder (required for linear models) and assembles everything into a single feature vector.

**Pipeline stages (8 total):** 3× StringIndexer + 3× OneHotEncoder + VectorAssembler + LinearRegression

In [8]:
# --- Pipeline stages for Linear Regression ---

# Categorical encoding: StringIndexer -> OneHotEncoder for each
lr_product_indexer = StringIndexer(inputCol="product", outputCol="product_idx")
lr_product_encoder = OneHotEncoder(inputCol="product_idx", outputCol="product_vec")

lr_region_indexer = StringIndexer(inputCol="region", outputCol="region_idx")
lr_region_encoder = OneHotEncoder(inputCol="region_idx", outputCol="region_vec")

lr_city_indexer = StringIndexer(inputCol="city", outputCol="city_idx")
lr_city_encoder = OneHotEncoder(inputCol="city_idx", outputCol="city_vec")

# Assemble all features into one vector
lr_assembler = VectorAssembler(
    inputCols=["quantity", "salary", "budget", "tenure_days",
               "product_vec", "region_vec", "city_vec"],
    outputCol="features"
)

# The estimator
lr = LinearRegression(
    featuresCol="features",
    labelCol="amount",
    maxIter=20,
    regParam=0.01
)

# Chain it all together
lr_pipeline = Pipeline(stages=[
    lr_product_indexer, lr_product_encoder,
    lr_region_indexer, lr_region_encoder,
    lr_city_indexer, lr_city_encoder,
    lr_assembler,
    lr
])

# Show the pipeline stages
print(f"Linear Regression Pipeline: {len(lr_pipeline.getStages())} stages")
for i, stage in enumerate(lr_pipeline.getStages()):
    print(f"  [{i}] {type(stage).__name__}")

Linear Regression Pipeline: 8 stages
  [0] StringIndexer
  [1] OneHotEncoder
  [2] StringIndexer
  [3] OneHotEncoder
  [4] StringIndexer
  [5] OneHotEncoder
  [6] VectorAssembler
  [7] LinearRegression


In [9]:
# Fit the pipeline on training data
lr_model = lr_pipeline.fit(train)

# Predict on test data
lr_predictions = lr_model.transform(test)

# Show sample predictions vs actuals
lr_predictions.select("product", "region", "quantity", "amount",
                       spark_round("prediction", 2).alias("predicted")) \
    .show(15, truncate=False)

+--------+-------+--------+------+---------+
|product |region |quantity|amount|predicted|
+--------+-------+--------+------+---------+
|Gadget Z|West   |1       |3200.0|5219.24  |
|Widget A|West   |15      |3750.0|4361.1   |
|Gadget Z|West   |1       |3200.0|5348.83  |
|Widget A|West   |10      |2500.0|2846.8   |
|Gadget X|West   |3       |6300.0|2980.43  |
|Gadget Z|East   |1       |3200.0|5438.56  |
|Widget A|East   |5       |1250.0|1318.32  |
|Widget A|East   |10      |2500.0|2864.25  |
|Widget B|East   |3       |890.5 |852.21   |
|Gadget Y|Central|3       |2670.0|2102.57  |
|Gadget Z|Central|1       |3200.0|5301.19  |
|Widget A|Central|2       |625.0 |11.79    |
|Widget A|South  |12      |3125.0|4735.96  |
|Widget A|Central|2       |625.0 |-197.45  |
|Widget B|South  |1       |296.83|1358.95  |
+--------+-------+--------+------+---------+
only showing top 15 rows


In [10]:
evaluator_rmse = RegressionEvaluator(labelCol="amount", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="amount", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="amount", metricName="mae")

lr_rmse = evaluator_rmse.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)
lr_mae = evaluator_mae.evaluate(lr_predictions)

print("=" * 45)
print("  LINEAR REGRESSION BASELINE RESULTS")
print("=" * 45)
print(f"  RMSE:  ${lr_rmse:,.2f}")
print(f"  MAE:   ${lr_mae:,.2f}")
print(f"  R\u00b2:    {lr_r2:.4f}")
print("=" * 45)

  LINEAR REGRESSION BASELINE RESULTS
  RMSE:  $1,471.21
  MAE:   $1,151.03
  R²:    0.0552


---
## 5. Improved Model: Gradient-Boosted Trees

Gradient-Boosted Trees (GBT) build an ensemble of decision trees sequentially, where each new tree corrects the errors of the previous ones. Key advantages over linear models:

- Captures **non-linear relationships** (e.g., quantity × product interactions)
- Handles categorical indices directly (**no OneHotEncoding needed**)
- Provides **feature importance** scores
- Generally achieves lower error on tabular data

We build a separate pipeline with simpler encoding (StringIndexer only, no OneHot) — **5 stages instead of 8**.

In [11]:
# --- Pipeline stages for GBT (simpler -- no OneHotEncoding needed) ---

gbt_product_indexer = StringIndexer(inputCol="product", outputCol="product_idx")
gbt_region_indexer = StringIndexer(inputCol="region", outputCol="region_idx")
gbt_city_indexer = StringIndexer(inputCol="city", outputCol="city_idx")

gbt_assembler = VectorAssembler(
    inputCols=["quantity", "salary", "budget", "tenure_days",
               "product_idx", "region_idx", "city_idx"],
    outputCol="features"
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="amount",
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_pipeline = Pipeline(stages=[
    gbt_product_indexer,
    gbt_region_indexer,
    gbt_city_indexer,
    gbt_assembler,
    gbt
])

# Compare pipeline complexity
print(f"Linear Regression pipeline: {len(lr_pipeline.getStages())} stages (needs OneHot)")
print(f"GBT pipeline:              {len(gbt_pipeline.getStages())} stages (index-only)")
print("\nGBT Pipeline Stages:")
for i, stage in enumerate(gbt_pipeline.getStages()):
    print(f"  [{i}] {type(stage).__name__}")

Linear Regression pipeline: 8 stages (needs OneHot)
GBT pipeline:              5 stages (index-only)

GBT Pipeline Stages:
  [0] StringIndexer
  [1] StringIndexer
  [2] StringIndexer
  [3] VectorAssembler
  [4] GBTRegressor


---
## 6. Hyperparameter Tuning with CrossValidator

Instead of guessing parameters, we define a **search grid** and let Spark evaluate every combination using k-fold cross-validation. Each fold trains on a different data subset, producing a robust estimate of model quality.

**This is where Spark’s parallelism pays off** — each fold and parameter combination can run in parallel across cores.

In [12]:
# Define the hyperparameter search space
param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [3, 5, 7]) \
    .addGrid(gbt.maxIter, [20, 50, 100]) \
    .addGrid(gbt.stepSize, [0.05, 0.1, 0.2]) \
    .build()

# 3-fold cross-validation
cv = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=RegressionEvaluator(labelCol="amount", metricName="rmse"),
    numFolds=3,
    seed=42
)

total_models = len(param_grid) * 3  # combos x folds
print(f"Parameter combinations: {len(param_grid)}")
print(f"Cross-validation folds: 3")
print(f"Total models to train:  {total_models}")
print(f"\nSearch space:")
print(f"  maxDepth:  [3, 5, 7]")
print(f"  maxIter:   [20, 50, 100]")
print(f"  stepSize:  [0.05, 0.1, 0.2]")

Parameter combinations: 27
Cross-validation folds: 3
Total models to train:  81

Search space:
  maxDepth:  [3, 5, 7]
  maxIter:   [20, 50, 100]
  stepSize:  [0.05, 0.1, 0.2]


In [13]:
# Fit CrossValidator (trains all 81 models)
cv_start = time.time()
cv_model = cv.fit(train)
cv_elapsed = time.time() - cv_start

# Extract the best model's GBT stage
best_gbt = cv_model.bestModel.stages[-1]

print(f"Cross-validation completed in {cv_elapsed:.1f}s ({total_models} models)")
print(f"\nBest model parameters:")
print(f"  maxDepth:  {best_gbt.getMaxDepth()}")
print(f"  numTrees:  {best_gbt.getNumTrees}")
print(f"  stepSize:  {best_gbt.getStepSize()}")
print(f"\nBest CV RMSE: ${min(cv_model.avgMetrics):,.2f}")

Cross-validation completed in 257.9s (81 models)

Best model parameters:
  maxDepth:  3
  numTrees:  100
  stepSize:  0.1

Best CV RMSE: $846.48


In [ ]:
# Predict with the tuned GBT model
gbt_predictions = cv_model.transform(test)

# Show predictions
gbt_predictions.select("product", "region", "quantity", "amount",
                        spark_round("prediction", 2).alias("predicted")) \
    .show(15, truncate=False)

# Compute metrics
gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)
gbt_mae = evaluator_mae.evaluate(gbt_predictions)

print("=" * 45)
print("  TUNED GBT RESULTS")
print("=" * 45)
print(f"  RMSE:  ${gbt_rmse:,.2f}")
print(f"  MAE:   ${gbt_mae:,.2f}")
print(f"  R\u00b2:    {gbt_r2:.4f}")
print("=" * 45)

---
## 7. Model Comparison

In [ ]:
print(f"{'Metric':<12} {'Linear Regression':>20} {'GBT (Tuned)':>20} {'Improvement':>15}")
print("-" * 70)

rmse_imp = (lr_rmse - gbt_rmse) / lr_rmse * 100 if lr_rmse != 0 else 0
mae_imp = (lr_mae - gbt_mae) / lr_mae * 100 if lr_mae != 0 else 0
r2_imp = (gbt_r2 - lr_r2) / max(abs(lr_r2), 0.0001) * 100

print(f"{'RMSE':<12} {'${:,.2f}'.format(lr_rmse):>20} {'${:,.2f}'.format(gbt_rmse):>20} {rmse_imp:>14.1f}%")
print(f"{'MAE':<12} {'${:,.2f}'.format(lr_mae):>20} {'${:,.2f}'.format(gbt_mae):>20} {mae_imp:>14.1f}%")
print(f"{'R\u00b2':<12} {lr_r2:>20.4f} {gbt_r2:>20.4f} {r2_imp:>14.1f}%")

---
## 8. Visual Analysis

In [ ]:
# Chart 1: Actual vs Predicted scatter plot
gbt_pdf = gbt_predictions.select("amount", "prediction").toPandas()

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(gbt_pdf["amount"], gbt_pdf["prediction"],
           alpha=0.7, edgecolors="k", linewidth=0.5, s=60)

# 45-degree reference line (perfect predictions)
min_val = min(gbt_pdf["amount"].min(), gbt_pdf["prediction"].min())
max_val = max(gbt_pdf["amount"].max(), gbt_pdf["prediction"].max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect prediction")

ax.set_xlabel("Actual Amount ($)", fontsize=12)
ax.set_ylabel("Predicted Amount ($)", fontsize=12)
ax.set_title("GBT Model: Predicted vs Actual Sale Amount", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Feature importances
feature_names = ["quantity", "salary", "budget", "tenure_days",
                 "product_idx", "region_idx", "city_idx"]
importances = best_gbt.featureImportances.toArray()

# Sort by importance
sorted_pairs = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)
sorted_names, sorted_importances = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(range(len(sorted_names)), sorted_importances,
               color="#4C72B0", edgecolor="k", linewidth=0.5)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=11)
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_title("GBT Feature Importances", fontsize=14)
ax.invert_yaxis()
ax.grid(True, axis="x", alpha=0.3)

for bar, val in zip(bars, sorted_importances):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Model comparison -- the "money slide"
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = ["Linear\nRegression", "GBT\n(Tuned)"]
colors = ["#D65F5F", "#4C72B0"]

# RMSE comparison
rmse_values = [lr_rmse, gbt_rmse]
axes[0].bar(models, rmse_values, color=colors, edgecolor="k", linewidth=0.5, width=0.5)
axes[0].set_ylabel("RMSE ($)", fontsize=12)
axes[0].set_title("RMSE (Lower is Better)", fontsize=13)
for i, v in enumerate(rmse_values):
    axes[0].text(i, v + max(rmse_values) * 0.02, f"${v:,.0f}",
                 ha="center", fontsize=11, fontweight="bold")

# R-squared comparison
r2_values = [max(lr_r2, 0), max(gbt_r2, 0)]  # Clamp negative R2 to 0 for display
axes[1].bar(models, r2_values, color=colors, edgecolor="k", linewidth=0.5, width=0.5)
axes[1].set_ylabel("R\u00b2", fontsize=12)
axes[1].set_title("R-Squared (Higher is Better)", fontsize=13)
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(r2_values):
    axes[1].text(i, v + 0.03, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Model Comparison: Baseline vs Tuned", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Scale-Up: From 100 Rows to 500K

Everything above ran on **100 rows** — trivial for any tool. Spark’s real power is **distributing work across partitions and cores** so that scaling to millions of rows doesn’t mean proportionally longer runtimes.

Let’s prove it:
1. Generate 500K synthetic rows **inside Spark** (no pandas, no files)
2. Time the same GBT pipeline on small vs large data
3. Show how repartitioning affects training speed

In [ ]:
# Generate 500K synthetic sales rows using Spark SQL functions
products = ["Widget A", "Widget B", "Widget C", "Gadget X", "Gadget Y", "Gadget Z"]
regions = ["Central", "East", "West", "South"]
cities = ["San Francisco", "New York", "Chicago"]

large_data = (
    spark.range(500_000)
    .withColumn("quantity", (floor(rand(seed=42) * 15) + 1).cast("int"))
    .withColumn("product", element_at(
        array(*[lit(p) for p in products]),
        (floor(rand(seed=43) * 6) + 1).cast("int")))
    .withColumn("region", element_at(
        array(*[lit(r) for r in regions]),
        (floor(rand(seed=44) * 4) + 1).cast("int")))
    .withColumn("city", element_at(
        array(*[lit(c) for c in cities]),
        (floor(rand(seed=45) * 3) + 1).cast("int")))
    .withColumn("salary", (floor(rand(seed=46) * 55000) + 60000).cast("int"))
    .withColumn("budget", (floor(rand(seed=47) * 1700000) + 800000).cast("int"))
    .withColumn("tenure_days", (floor(rand(seed=48) * 3000) + 100).cast("int"))
    .withColumn("amount", expr("quantity * (300 + rand(49) * 600)"))
    .drop("id")
)

# Cache it so generation cost isn't counted in training time
large_data.cache()

print(f"Generated: {large_data.count():,} rows")
print(f"Partitions: {large_data.rdd.getNumPartitions()}")
print()
large_data.show(5, truncate=False)

In [ ]:
# Time the GBT pipeline on small data (100 rows) vs large data (500K rows)

# Small data timing
small_data = data.cache()
small_data.count()  # Force materialization

start = time.time()
gbt_pipeline.fit(small_data)
small_time = time.time() - start

# Large data timing
start = time.time()
gbt_pipeline.fit(large_data)
large_time = time.time() - start

ratio = large_data.count() / small_data.count()
time_ratio = large_time / small_time if small_time > 0 else float('inf')

print(f"{'Dataset':<20} {'Rows':>10} {'Train Time':>12} {'Rows/sec':>12}")
print("-" * 56)
print(f"{'Small (original)':<20} {small_data.count():>10,} {small_time:>11.1f}s {small_data.count()/small_time:>11,.0f}")
print(f"{'Large (synthetic)':<20} {large_data.count():>10,} {large_time:>11.1f}s {large_data.count()/large_time:>11,.0f}")
print(f"\nData is {ratio:,.0f}x larger, but training is only {time_ratio:.1f}x slower.")
print(f"Throughput (rows/sec) INCREASED because Spark parallelizes across partitions.")

In [ ]:
# Show how partition count affects training speed
# More partitions = more parallel tasks (up to available cores)
partition_counts = [1, 2, 4, 8]
partition_times = []

print(f"Training GBT pipeline on 500K rows with varying partition counts:")
print(f"(Available cores: local[*])\n")

for n in partition_counts:
    repartitioned = large_data.repartition(n).cache()
    repartitioned.count()  # Force materialization
    
    start = time.time()
    gbt_pipeline.fit(repartitioned)
    elapsed = time.time() - start
    partition_times.append(elapsed)
    
    repartitioned.unpersist()
    print(f"  {n:>2} partition(s): {elapsed:>6.1f}s")

print(f"\nSpeedup from 1 to {partition_counts[-1]} partitions: {partition_times[0]/partition_times[-1]:.1f}x")

In [ ]:
# Chart 4: Training time vs number of partitions
fig, ax = plt.subplots(figsize=(8, 5))

bar_colors = ["#D65F5F" if t == max(partition_times) else "#4C72B0" for t in partition_times]
bars = ax.bar([str(n) for n in partition_counts], partition_times,
              color=bar_colors, edgecolor="k", linewidth=0.5, width=0.5)

for bar, t in zip(bars, partition_times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{t:.1f}s", ha="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Number of Partitions", fontsize=12)
ax.set_ylabel("Training Time (seconds)", fontsize=12)
ax.set_title("GBT Training Time vs Partition Count (500K rows)", fontsize=14)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight**: Spark’s value is NOT speed on small data — the JVM startup and task scheduling overhead actually makes it slower than scikit-learn on 100 rows. At scale (hundreds of thousands to billions of rows), distributing work across partitions means:

- **More partitions → more parallel tasks** (up to your core/executor count)
- **Sub-linear scaling** — 5,000× more data does NOT mean 5,000× longer training
- **Throughput increases** as data grows, because fixed overhead is amortized

In production clusters with hundreds of nodes, this same pattern extends to petabyte-scale data.

---
## 10. Model Persistence: Save, Load, Verify

In production, you train once and serve predictions many times. MLlib supports saving the **entire fitted pipeline** (encoders + model) to disk and loading it in a separate application.

In [ ]:
# Save the best pipeline model (includes all fitted stages)
model_path = "../output/gbt_sales_model"
cv_model.bestModel.write().overwrite().save(model_path)
print(f"Model saved to: {model_path}")

# Load it back (in production, this would be in a separate process)
loaded_model = PipelineModel.load(model_path)
print(f"Model loaded: {len(loaded_model.stages)} stages")

# Verify: predictions from loaded model match original
loaded_predictions = loaded_model.transform(test)
loaded_rmse = evaluator_rmse.evaluate(loaded_predictions)

print(f"\nOriginal RMSE:  ${gbt_rmse:,.2f}")
print(f"Loaded RMSE:    ${loaded_rmse:,.2f}")
print(f"Match:          {abs(gbt_rmse - loaded_rmse) < 0.01}")

In [ ]:
# Simulate scoring new records -- the pipeline handles all encoding automatically
new_sales = spark.createDataFrame([
    ("Widget A", 5, "West", "San Francisco", 110000, 2500000, 2950),
    ("Gadget Z", 2, "East", "Chicago", 76000, 1800000, 1800),
    ("Gadget X", 1, "Central", "New York", 78000, 1200000, 1600),
], ["product", "quantity", "region", "city", "salary", "budget", "tenure_days"])

scored = loaded_model.transform(new_sales)
scored.select("product", "quantity", "region", "city",
              spark_round("prediction", 2).alias("predicted_amount")).show(truncate=False)

---
## Key Takeaways

**What MLlib handles for you:**
- Feature encoding (StringIndexer, OneHotEncoder) — no manual label maps
- Feature assembly (VectorAssembler) — pack any number of columns into a single vector
- Pipeline composition — chain transformers and estimators into one reproducible unit
- Hyperparameter tuning (CrossValidator + ParamGridBuilder) — exhaustive search with k-fold validation
- Model serialization — save/load the entire fitted pipeline, not just the model weights

**Design decisions that matter:**
- **Encoding strategy depends on the model**: OneHot for linear models, index-only for trees
- **Pipeline over manual steps**: ensures train and test data get identical preprocessing
- **Cross-validation over hold-out**: more reliable performance estimates, especially on small data

**Parallelism lessons:**
- Spark’s overhead makes it slower than single-machine tools on tiny datasets
- At scale, **more partitions = more parallelism** (up to available cores)
- Training throughput (rows/sec) **increases** as data grows
- In production, this scales horizontally across hundreds of executor nodes

**What we did NOT cover** (topics for a deeper dive):
- Feature selection and dimensionality reduction (PCA, ChiSqSelector)
- Streaming prediction (Structured Streaming + MLlib)
- Model serving at scale (MLflow, Spark model export to ONNX/PMML)
- Custom transformers (extending `Transformer` / `Estimator`)

In [ ]:
# Clean up
large_data.unpersist()
small_data.unpersist()
spark.stop()
print("SparkSession stopped.")